In [18]:
orcid = '0000-0002-8804-0722' # Fill your orcid here

In [19]:
import requests

We use the `/works` api to list all works related to the orcid. This gives a summary of all works, so citation information is not included. We collect the `put-code` of all works to retrieve the citation information later.

In [20]:
response = requests.get('https://pub.orcid.org/v3.0/{}/works'.format(orcid),
                        headers={"Accept": "application/orcid+json" })
record = response.json()

In [21]:
put_codes = []
for work in record['group']:
    put_code = work['work-summary'][0]['put-code']
    put_codes.append(put_code)
put_code = put_codes[0]

We use the `/<orcid>/work/<put-code>` endpoint to retrieve the citation information for each record.

In [22]:
def work_to_bibtex(work):
    # Map ORCID types to BibTeX entry types
    type_map = {
        'journal-article': 'article',
        'book': 'book',
        'conference-paper': 'inproceedings'
    }
    entry_type = type_map.get(work.get('type'), 'misc')

    # Build author list safely
    contributors = (work.get('contributors') or {}).get('contributor', [])
    authors = []
    for c in contributors:
        name = (c.get('credit-name') or {}).get('value')
        if name:
            authors.append(name)
    author_str = ' and '.join(authors) if authors else 'Unknown'

    # Safely extract title
    title_dict = work.get('title') or {}
    title_subdict = title_dict.get('title') or {}
    title = title_subdict.get('value', 'No Title')

    # Safely extract year
    pub_date = work.get('publication-date') or {}
    year = (pub_date.get('year') or {}).get('value', '????')

    # Safely extract DOI
    external_ids = (work.get('external-ids') or {}).get('external-id', [])
    doi = next((eid.get('external-id-value') for eid in external_ids if eid.get('external-id-type') == 'doi'), None)

    # Safely extract URL
    url = (work.get('url') or {}).get('value')

    # Safely extract journal/booktitle
    journal_title = (work.get('journal-title') or {}).get('value')
    fields = {
        'author': author_str,
        'title': title,
        'year': year,
        'doi': doi,
        'url': url
    }
    if entry_type == 'article':
        fields['journal'] = journal_title
    elif entry_type == 'inproceedings':
        fields['booktitle'] = journal_title

    # Generate BibTeX key
    key = (authors[0].replace(',', ' ').split()[0] if authors else 'Unknown') + year + (title.split()[0] if title != 'No Title' else '')

    # Build BibTeX entry
    bibtex = f"@{entry_type}{{{key},\n"
    for k, v in fields.items():
        if v:
            bibtex += f"  {k} = {{{v}}},\n"
    bibtex = bibtex.rstrip(',\n') + "\n}\n"
    return bibtex

In [23]:
import re
import requests
from bs4 import BeautifulSoup

def fetch_unipd_metadata(url):
    response = requests.get(url, headers={'User-Agent': 'Foo'})
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    # Only look for meta tags inside the body
    meta_tags = soup.find_all('meta')
    meta = {tag.get('name'): tag.get('content', '') for tag in meta_tags if tag.get('name')}

    skip_type = "282"
    type_full = meta.get('DC.type', '')
    if type_full == skip_type:
        return None

    type_map = {
    '262': 'article',
    '263': 'article',
    '264': 'article',
    '265': 'article',
    '266': 'article',
    '267': 'article',
    '268': 'article',
    '269': 'article',
    '270': 'article',
    '273': 'inproceedings',
    '278': 'incollection',
    '279': 'incollection',
    '280': 'book',
    '281': 'book',
    '282': 'inproceedings',
    '283': 'inproceedings',
    '284': 'inproceedings',
    '285': 'inproceedings',
    '286': 'inproceedings',
    '298': 'phdthesis',
    '299': 'phdthesis',
    '300': 'mastersthesis',
    '301': 'mastersthesis',
    '302': 'bachelorsthesis',
    '303': 'bachelorsthesis',
    '304': 'report',
    '305': 'manual',
    '306': 'misc',
}
    bib_type = 'misc'
    for k, v in type_map.items():
        if type_full.startswith(k):
            bib_type = v
            break

    authors = meta.get('DC.description', 'Unknown')
    authors = ' and '.join([a.strip() for a in authors.split(';') if a.strip()])

    return {
        'title': meta.get('DC.title', 'No Title'),
        'author': authors,
        'year': meta.get('DCTERMS.issued', '????')[:4],
        'journal': meta.get('citation_journal_title', ''),
        'conference': meta_tags[16].get('content', ''),
        'type': bib_type
    }

def update_bib_entries_with_unipd(entries):
    updated_entries = []
    url_pattern = re.compile(r'url\s*=\s*\{(https?://www\.research\.unipd\.it/[^}]+)\}', re.IGNORECASE)
    type_pattern = re.compile(r'^@(\w+)\{', re.MULTILINE)
    for entry in entries:
        match = url_pattern.search(entry)
        if match:
            url = match.group(1)
            metadata = fetch_unipd_metadata(url)
            if metadata is None:
                updated_entries.append(entry)  # Skip update for this entry
                continue
            entry = type_pattern.sub(f"@{metadata['type']}{{", entry, count=1)
            entry = re.sub(r'author\s*=\s*\{[^}]*\}', f"author = {{{metadata['author']}}}", entry)
            entry = re.sub(r'title\s*=\s*\{[^}]*\}', f"title = {{{metadata['title']}}}", entry)
            entry = re.sub(r'year\s*=\s*\{[^}]*\}', f"year = {{{metadata['year']}}}", entry)
            entry = re.sub(r'journal\s*=\s*\{[^}]*\}', f"journal = {{{metadata['journal']}}}", entry)
            entry = re.sub(r'booktitle\s*=\s*\{[^}]*\}', f"booktitle = {{{metadata['journal']}}}", entry)
            # update key if author or year changed
            key_match = re.search(r'@(\w+)\{([^,]+),', entry)
            if key_match:
                old_key = key_match.group(2)
                new_key = (metadata['author'].split(',')[0] if metadata['author'] != 'Unknown' else 'Unknown') + metadata['year']
                entry = entry.replace(old_key, new_key, 1)

        updated_entries.append(entry)
    return updated_entries

In [24]:
books = []
journals = []
conferences = []
others = []
for put_code in put_codes:
    response = requests.get('https://pub.orcid.org/v3.0/{}/work/{}'.format(orcid, put_code),
                            headers={"Accept": "application/orcid+json" })
    work = response.json()
    if work['type'] == 'journal-article':
        journals.append(work_to_bibtex(work))
    elif work['type'] == 'book':
        books.append(work_to_bibtex(work))
    elif work['type'] == 'conference-paper':
        conferences.append(work_to_bibtex(work))
    else:
        others.append(work_to_bibtex(work))

In [25]:
# try to update entries with unipd metadata
others = update_bib_entries_with_unipd(others)
journals = update_bib_entries_with_unipd(journals)
conferences = update_bib_entries_with_unipd(conferences)
books = update_bib_entries_with_unipd(books)

# sort them in the correct list if possible
def sort_entries(entries, journals, conferences, books):
    for entry in entries:
        if '@article' in entry:
            journals.append(entry)
            entries.remove(entry)
        elif '@inproceedings' in entry:
            conferences.append(entry)
            entries.remove(entry)
        elif '@book' in entry:
            books.append(entry)
            entries.remove(entry)
    return entries
others = sort_entries(others, journals, conferences, books)

In [26]:
with open('journal.bib', 'w') as bibfile:
    for citation in journals:
        bibfile.write(citation)
        bibfile.write('\n')
with open('conference.bib', 'w') as bibfile:
    for citation in conferences:
        bibfile.write(citation)
        bibfile.write('\n')
with open('book.bib', 'w') as bibfile:
    for citation in books:
        bibfile.write(citation)
        bibfile.write('\n')
with open('other.bib', 'w') as bibfile:
    for citation in others:
        bibfile.write(citation)
        bibfile.write('\n')
